# Implementing and Evaluating Greediness Metric(s) for Combo Lock

In [1]:
import plotly.express as px
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Any
from pprint import pprint as pp

In [2]:
styles = ["3", "4", "5", "6"]
efforts = ["default"]#,"medium","high"]
styles = [style + '_' + effort for effort in efforts for style in styles]

results_paths: List[str] = [
    f'../src/optimal_explorer/strategies/combination_lock/logs/game_results/style{style}.jsonl'
    for style in styles
]
bayes_optimal_path: str = f'../src/optimal_explorer/strategies/combination_lock/logs/bayes_optimal_2.jsonl'


models = [
    "Gemini Pro 2.5",
    "DeepSeek R1",
    "Claude Opus 4",
    "Claude 3.5 Sonnet",
    "OpenAI o3",
]

model_ids = [
    "google/gemini-2.5-pro-preview",
    "deepseek/deepseek-r1-0528",
    "anthropic/claude-opus-4",
    "anthropic/claude-3.5-sonnet",
    "openai/o3",
]

In [3]:
results = []

for style, results_path in zip(styles, results_paths):
    with open(results_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            pp(data)
            break
        break

{'game_id': 21,
 'history': [{'attempt': 1,
              'feedback': [0, 0, 0],
              'feedback_str': '⬜⬜⬜',
              'guess': '!@#'},
             {'attempt': 2,
              'feedback': [2, 2, 2],
              'feedback_str': '🟩🟩🟩',
              'guess': '$%^'}],
 'model': 'anthropic/claude-3.5-sonnet',
 'num_attempts': 2,
 'prompt_style': 3,
 'reasoning_effort': None,
 'success': True,
 'target_combination': '$%^',
 'timestamp': '2025-06-27T06:56:28.721216'}


In [4]:
results = []

for style, results_path in zip(styles, results_paths):
    with open(results_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            regret = [0 if z['feedback'] == [2, 2, 2] else 1 for z in data['history']] + [0] * (12 - len(data['history']))
            model = data['model']
            results.append({
                'game_id': data['game_id'],
                'model': model + f' (s={style})',
                'regret': regret,
                'length': len(data['history']),
                'style': style,
                'history': data['history'],
                'num_attempts': data['num_attempts'],
                'target_combination': data['target_combination'],
            })
results_df = pd.DataFrame(results)

bayes_optimal = []

with open(bayes_optimal_path, 'r') as file:
    for line in file:
        data = json.loads(line)
        regret = [0 if z['feedback'] == [2, 2, 2] else 1 for z in data['history']] + [0] * (12 - len(data['history']))
        bayes_optimal.append({
            'game_id': data['game_id'],
            'model': 'Bayes Optimal',
            'regret': regret,
            'length': len(data['history']),
            'style': style
        })
bayes_optimal_df = pd.DataFrame(bayes_optimal)

In [5]:
results_df = results_df.drop_duplicates(subset=['game_id', 'model'], keep='last')
bayes_optimal_df = bayes_optimal_df.drop_duplicates(subset=['game_id', 'model'], keep='last')

In [6]:
results_df

,game_id,model,regret,length,style,history,num_attempts,target_combination
0,21,anthropic/claude-3.5-sonnet (s=3_default),"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",2,3_default,"[{'attempt': 1, 'guess': '!@#', 'feedback': [0...",2,$%^
1,15,anthropic/claude-3.5-sonnet (s=3_default),"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",3,3_default,"[{'attempt': 1, 'guess': '!@#', 'feedback': [0...",3,%@#
2,36,anthropic/claude-3.5-sonnet (s=3_default),"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",3,3_default,"[{'attempt': 1, 'guess': '!@#', 'feedback': [0...",3,&*#
3,21,anthropic/claude-opus-4 (s=3_default),"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",2,3_default,"[{'attempt': 1, 'guess': '!@#', 'feedback': [0...",2,$%^
4,60,anthropic/claude-3.5-sonnet (s=3_default),"[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]",5,3_default,"[{'attempt': 1, 'guess': '!@#', 'feedback': [1...",5,&!@
...,...,...,...,...,...,...,...,...
1995,75,deepseek/deepseek-r1-0528 (s=6_default),"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",12,6_default,"[{'attempt': 1, 'guess': '!p5', 'feedback': [0...",12,p57
1996,78,deepseek/deepseek-r1-0528 (s=6_default),"[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]",10,6_default,"[{'attempt': 1, 'guess': '!p5', 'feedback': [0...",10,86%
1997,5,deepseek/deepseek-r1-0528 (s=6_default),"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",12,6_default,"[{'attempt': 1, 'guess': '8q5', 'feedback': [0...",12,5#%
1998,8,deepseek/deepseek-r1-0528 (s=6_default),"[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]",8,6_default,"[{'attempt': 1, 'guess': '!p5', 'feedback': [0...",8,%*^


In [7]:
results_df.columns

Index(['game_id', 'model', 'regret', 'length', 'style', 'history',
       'num_attempts', 'target_combination'],
      dtype='object')

In [8]:
results_df.iloc[42]['history']

[{'attempt': 1, 'guess': '!@#', 'feedback': [0, 0, 0], 'feedback_str': '⬜⬜⬜'},
 {'attempt': 2, 'guess': '$%^', 'feedback': [0, 2, 2], 'feedback_str': '⬜🟩🟩'},
 {'attempt': 3, 'guess': 'p%^', 'feedback': [0, 2, 2], 'feedback_str': '⬜🟩🟩'},
 {'attempt': 4, 'guess': 'q%^', 'feedback': [2, 2, 2], 'feedback_str': '🟩🟩🟩'}]

In [9]:
def compute_greediness(results_df):
    import numpy as np

    # Collect per-game greediness for each model
    model_greediness = {}

    for _, row in results_df.iterrows():
        model = row['model']
        history = row['history']

        total_predictions = 0
        correct_predictions = 0

        for entry in history:
            feedback = entry['feedback']
            total_predictions += len(feedback)
            correct_predictions += sum(1 for f in feedback if f == 2)

        if model not in model_greediness:
            model_greediness[model] = []

        # Per-game greediness (fraction correct)
        greediness = correct_predictions / total_predictions if total_predictions > 0 else 0
        model_greediness[model].append(greediness)

    # Build DataFrame with mean and std for error bars
    data = []
    for model, greediness_list in model_greediness.items():
        greediness_mean = np.mean(greediness_list)
        greediness_std = np.std(greediness_list)
        data.append({
            'model': model,
            'greediness_mean': greediness_mean,
            'greediness_std': greediness_std
        })

    return pd.DataFrame(data)

In [14]:
greediness_df = compute_greediness(results_df)

In [15]:
# Split the model column into model and style
greediness_df[['model_base', 'style']] = greediness_df['model'].str.extract(r'(.+?)\s*\(s=(\d+_default)\)')
greediness_df = greediness_df[['model_base', 'style', 'greediness_mean', 'greediness_std']].rename(columns={'model_base': 'model'})
# remove the _default from the style column
greediness_df['style'] = greediness_df['style'].str.replace('_default', '')

In [16]:
greediness_df

,model,style,greediness_mean,greediness_std
0,anthropic/claude-3.5-sonnet,3,0.412810,0.130127
1,anthropic/claude-opus-4,3,0.435235,0.126826
2,openai/o3,3,0.453437,0.119838
3,google/gemini-2.5-pro-preview,3,0.383318,0.155580
4,deepseek/deepseek-r1-0528,3,0.342073,0.113637
5,anthropic/claude-3.5-sonnet,4,0.450150,0.118863
6,anthropic/claude-opus-4,4,0.454086,0.133267
7,google/gemini-2.5-pro-preview,4,0.445425,0.112657
8,openai/o3,4,0.474889,0.113266
9,deepseek/deepseek-r1-0528,4,0.377328,0.122309


In [20]:
import plotly.express as px
import plotly.io as pio

# dark colors
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
models = results_df['model'].unique()

# Set default font to LaTeX-style serif
pio.templates["latex_white"] = pio.templates["plotly_white"].update({
    "layout": {
        "font": {
            "family": "Latin Modern Roman, Times New Roman, serif",
            "size": 16
        },
        "legend": {
            "bgcolor": "white",
            "bordercolor": "black",
            "borderwidth": 1,
            "x": 0.2,
            "y": 0.01
        },
        "xaxis": {
            "showgrid": True,
            "gridcolor": "#e5e5e5",
            "title_font": {"size": 18}
        },
        "yaxis": {
            "showgrid": True,
            "gridcolor": "#e5e5e5",
            "title_font": {"size": 18}
        },
        "plot_bgcolor": "white",
        "paper_bgcolor": "white"
    }
})
pio.templates.default = "latex_white"

# Bar chart with error bars
fig = px.bar(
    greediness_df,
    x="style",
    y="greediness_mean",
    color="model",
    barmode="group",
    error_y="greediness_std",
    color_discrete_sequence=colors,
    title=""
)

fig.update_layout(
    width=800,
    height=500,
    legend_title_text="",
    title_font=dict(size=22),
)

fig.update_xaxes(title_text="Prompt Style")
fig.update_yaxes(title_text="Greediness")

# yrange
fig.update_yaxes(range=[0, 0.7], dtick=0.1)

fig.show()